# Lekce 05 - Agentní RAG


## Nastavení

Tento notebook ukazuje vzor Agentic RAG (Retrieval-Augmented Generation) pomocí Microsoft Agent Framework.

**Požadavky:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — váš koncový bod služby Azure AI Search
- `AZURE_SEARCH_API_KEY` — váš klíč API pro Azure AI Search
- nasazení Azure OpenAI nakonfigurované pomocí proměnných prostředí
- ověřený Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Co je Agentic RAG?

Tradiční RAG následuje pevně daný proces: vyhledat dokumenty a poté vygenerovat odpověď. **Agentic RAG** jde dále tím, že agentovi dává autonomii rozhodovat **kdy** a **jak** získávat informace.

S Agentic RAG může agent:
- **Rozhodnout**, zda je před zodpovězením otázky potřeba vyhledávání
- **Vybrat**, který zdroj dat nebo nástroj použít pro dotaz
- **Zhodnotit** získané výsledky a provést následná vyhledávání, pokud první pokus nestačí
- **Kombinovat** informace z několika kroků vyhledávání do koherentní odpovědi

To činí agenta pružnějším a přesnějším v porovnání se statickým procesem získání a poté generování.


## Vytvoření vyhledávacího nástroje

V Agentic RAG jsou externí zdroje dat zabaleny jako **nástroje**, které může agent vyvolat na vyžádání. To umožňuje agentovi považovat vyhledávání za další akci, kterou může provést, místo povinného kroku.

Níže definujeme cestovní znalostní bázi a zpřístupníme ji jako nástroj, který může agent volat pro vyhledání informací o destinacích.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Vytváření RAG agenta

Nyní vytvoříme agenta, který je instruován, aby **vždy před odpovědí získal informace**. Agent používá nástroj `search_travel_knowledge`, aby zakotvil své odpovědi ve znalostní bázi, místo aby se spoléhal na svá vlastní tréninková data.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterativní vyhledávání — Vzor Maker-Checker

Klíčovou výhodou Agentic RAG je **iterativní vyhledávání**. Agent může provádět několik kol vyhledávání, aby ověřil, upřesnil nebo rozšířil své počáteční nálezy — podobně jako pracovní postup „maker-checker“:

1. **Maker krok**: Agent vyhledá počáteční informace a připraví odpověď.
2. **Checker krok**: Agent provádí další vyhledávání, aby ověřil podrobnosti nebo vyplnil mezery.

Níže je agentovi položena otázka, která vyžaduje porovnání více destinací, což ho podněcuje k několikerému vyhledávání.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Shrnutí

V této lekci jste se naučili, jak postavit systém **Agentic RAG** pomocí Microsoft Agent Framework:

- **Agentic RAG** umožňuje agentům autonomně rozhodovat, kdy vyhledávat informace, což činí vyhledávání dynamickým místo pevně daného.
- **Nástroje jako zdroje dat**: Externí databáze znalostí (jako Azure AI Search) jsou zabalena jako nástroje, které agent může vyvolat.
- **Iterativní vyhledávání**: Vzor maker-checker umožňuje agentovi provádět více kol vyhledávání — vyhledávat, ověřovat a zpřesňovat — před vytvořením konečné odpovědi.

V produkčním prostředí byste místo paměťové `TRAVEL_KNOWLEDGE_BASE` použili skutečný index Azure AI Search pro zpracování rozsáhlého vyhledávání cestovních dokumentů.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
